## Title Entendimiento del problema y descarga de datos

### By:
Dovaribi Carupia Yagari

### Date:
2026-08-15

### Description:

Crear una nueva rama de git (Usar Gitflow) y Crear un notebook para la obtención de datos tipo RAW.

crear la rama a partir de un issue: https://docs.github.com/es/issues/tracking-your-work-with-issues/using-issues/creating-a-branch-for-an-issue

Realizar los pasos de: https://joserzapata.github.io/post/ciencia-datos-proyecto-python/1-data/

Descargar los datos y entender el problema a realizar, contestar en el notebook

¿Cual es el objetivo del problema?
¿Cómo se usará su solución?
¿Cuáles son las soluciones actuales (si las hay)?
¿Cómo se debe enmarcar este problema (supervisado / no supervisado, en línea / fuera de línea, etc.)
¿Cómo se debe medir el desempeño o el rendimiento de la solución, una primera intuicion?
¿La medida de desempeño está alineada con el objetivo del problema?
¿Cuál sería el desempeño o rendimiento mínimo necesario para alcanzar el objetivo del problema?
¿Cuáles son los problemas parecidos? ¿Se puede reutilizar experiencias o herramientas ya creadas?
¿Hay experiencia del problema disponible?
(Importante) ¿Cómo se puede resolver el problema manualmente?
Hacer un listado de los supuestos que hay hasta este momento.
Cual es la fuente de los datos?
Como se actualizan los datos?
Cada cuanto tiempo se actualizan los datos

## 📚 Import  libraries

In [1]:
import pandas as pd

## Entendimiento del problema

# 1. Ingestión y Entendimiento del Problema

**Objetivo:** Obtener los datos del diagnóstico de pacientes hepáticos en India (ILPD) y definir el marco de trabajo del problema bajo el enfoque de Ciencias de Datos en Producción.

---

## 1.1 Entendimiento del Problema y Contexto

Se han seleccionado y respondido las preguntas fundamentales de la metodología para establecer el alcance de esta Prueba de Concepto (POC):

**¿Cuál es el objetivo del problema?**
Desarrollar un modelo predictivo capaz de identificar tempranamente si un paciente padece una enfermedad hepática basándose en resultados de exámenes de sangre rutinarios (enzimas, proteínas, bilirrubina) y datos demográficos.

**¿Cómo se usará su solución?**
En un entorno real de MLOps, esta solución se desplegaría como una API REST o una aplicación web (ej. Streamlit/Gradio). Un médico ingresaría los resultados de laboratorio del paciente y el sistema devolvería una probabilidad de padecer la enfermedad para priorizar la atención o solicitar exámenes confirmatorios más invasivos (como biopsias). 

**¿Cómo se debe enmarcar este problema?**
* **Tipo de tarea:** Aprendizaje Supervisado (Clasificación Binaria).
* **Entorno de Inferencia:** En línea (On-demand) para predicciones individuales en la clínica.

**¿Cómo se debe medir el desempeño de la solución y está alineado con el negocio?**
Sí está alineado. En el ámbito médico, el costo de un **falso negativo** (decirle a alguien enfermo que está sano, retrasando su tratamiento) es el riesgo más crítico. Por lo tanto, la métrica principal a optimizar será el **Recall (Sensibilidad)** de la clase minoritaria (pacientes enfermos). Secundariamente, se observará el **F1-Score** y el **ROC-AUC** para asegurar que el modelo no genere demasiadas falsas alarmas (falsos positivos).

**¿Cómo se puede resolver el problema manualmente?**
El diagnóstico manual requiere que un especialista cruce múltiples indicadores: niveles altos de Fosfatasa Alcalina combinados con alteraciones en la Bilirrubina y las transaminasas (ALT/AST), ajustando el criterio clínico según la edad y género del paciente.

**Fuente y actualización de los datos:**
* **Fuente:** Indian Liver Patient Dataset (ILPD) en formato CSV estático.
* **Actualización:** Al ser una POC, los datos son estáticos (offline) y no tendrán actualizaciones periódicas. En un escenario productivo real, se requeriría un pipeline de ingesta batch diario desde la base de datos del hospital.

**Listado de los supuestos hasta este momento:**
1. En la variable `Dataset`, asumimos que el valor `1` indica enfermedad hepática y `2` indica paciente sin enfermedad.
2. Las características de los pacientes de la India son biológicamente generalizables como prueba de concepto para la validación del modelo base.

## 💾 Load data

In [4]:
import logging
from pathlib import Path
import pandas as pd

# 1. Configuración básica de logging para trazabilidad
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# 2. Definición de constantes (Rutas y Columnas)
# Ajusta el Path según la estructura de carpetas de tu repositorio
DATA_DIR = Path("../..") / "data" / "01_raw"
FILE_PATH = DATA_DIR / "Pacientes_porblemas_higado_india.csv"

# Definimos estrictamente las columnas para evitar errores de parseo por comas huérfanas
VALID_COLUMNS = [
    'Age', 'Gender', 'Total_Bilirubin', 'Direct_Bilirubin', 'Alkaline_Phosphotase', 
    'Alamine_Aminotransferase', 'Aspartate_Aminotransferase', 'Total_Protiens', 
    'Albumin', 'Albumin_and_Globulin_Ratio', 'Dataset'
]

def load_raw_data(file_path: Path, usecols: list[str]) -> pd.DataFrame:
    """
    Carga los datos crudos desde un archivo CSV.

    Args:
        file_path (Path): Ruta absoluta o relativa al archivo CSV.
        usecols (list[str]): Lista de nombres de columnas a extraer.

    Returns:
        pd.DataFrame: DataFrame de Pandas con los datos cargados.

    Raises:
        FileNotFoundError: Si el archivo no existe en la ruta especificada.
        ValueError: Si hay un error de lectura en el formato de los datos.
    """
    logging.info("Iniciando la carga de datos desde: %s", file_path)
    
    if not file_path.exists():
        logging.error("El archivo no se encontró: %s", file_path)
        raise FileNotFoundError(f"Ruta inválida o archivo inexistente: {file_path}")

    try:
        df = pd.read_csv(file_path, usecols=usecols)
        logging.info("Datos cargados exitosamente. Dimensiones: %s", df.shape)
        return df
    except Exception as e:
        logging.error("Error inesperado al leer el archivo CSV: %s", e)
        raise ValueError(f"Error procesando el archivo CSV: {e}") from e

# 3. Ejecución principal
if __name__ == "__main__":
    # Carga de datos
    df_raw = load_raw_data(FILE_PATH, VALID_COLUMNS)
    
    # Inspección rápida
    print("\n" + "="*40)
    print("ESTRUCTURA DEL DATASET")
    print("="*40)
    df_raw.info()
    
    display(df_raw.head())

2026-08-20 11:13:52,090 - INFO - Iniciando la carga de datos desde: ../../data/01_raw/Pacientes_porblemas_higado_india.csv
2026-08-20 11:13:52,106 - INFO - Datos cargados exitosamente. Dimensiones: (663, 11)



ESTRUCTURA DEL DATASET
<class 'pandas.DataFrame'>
RangeIndex: 663 entries, 0 to 662
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         662 non-null    float64
 1   Gender                      655 non-null    str    
 2   Total_Bilirubin             657 non-null    float64
 3   Direct_Bilirubin            656 non-null    float64
 4   Alkaline_Phosphotase        646 non-null    float64
 5   Alamine_Aminotransferase    644 non-null    float64
 6   Aspartate_Aminotransferase  651 non-null    float64
 7   Total_Protiens              656 non-null    float64
 8   Albumin                     661 non-null    float64
 9   Albumin_and_Globulin_Ratio  659 non-null    float64
 10  Dataset                     648 non-null    float64
dtypes: float64(10), str(1)
memory usage: 57.1 KB


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,65.0,Female,0.7,0.1,187.0,16.0,18.0,6.8,3.3,0.90,1.0
1,62.0,Male,10.9,5.5,699.0,64.0,100.0,7.5,3.2,0.74,1.0
2,62.0,Male,7.3,4.1,490.0,60.0,68.0,7.0,3.3,0.89,1.0
3,58.0,Male,1.0,0.4,182.0,14.0,20.0,6.8,3.4,1.00,1.0
4,72.0,Male,3.9,2.0,195.0,27.0,59.0,7.3,2.4,0.40,1.0
